# Modeling and Evaluating 

Do có nhiều sự lựa chọn nên ở đây tụi em quyết định chọn LightGBM do có hiệu năng và cũng là model mạnh nhất hiện tại

Sau nhiều lần thử nghiệm với các model khác thì thấy LightGBM hiệu suất hơn hẳn

- evaluating scale với wrmsse 
- tại sao lại chọn những tham số đó 
- lí do chọn lgbm - ưu chuộng và hiệu quả nhất 
- test với model khác - cân nhắc
- 

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import lightgbm as lgb
from lightgbm import early_stopping, log_evaluation
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OrdinalEncoder, LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
import os, sys, gc, time, warnings, pickle, psutil, random, optuna
print("Num GPUs Available:", len(tf.config.experimental.list_physical_devices('GPU')))
print("GPU Details:", tf.config.experimental.list_physical_devices('GPU'))

Num GPUs Available: 1
GPU Details: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
df = pd.read_parquet('../../data/train_val.parquet')
print(df.head())

def downcast(df):
    cols = df.dtypes.index.tolist()
    types = df.dtypes.values.tolist()
    
    for i, t in enumerate(types):
        col = cols[i]
        if 'int' in str(t):
            c_min = df[col].min()
            c_max = df[col].max()
            if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                df[col] = df[col].astype(np.int8)
            elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                df[col] = df[col].astype(np.int16)
            elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                df[col] = df[col].astype(np.int32)
            else:
                df[col] = df[col].astype(np.int64)

        elif 'float' in str(t):
            c_min = df[col].min()
            c_max = df[col].max()
            # ⚠️ Không dùng float16 vì không lưu được trong Parquet
            if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                df[col] = df[col].astype(np.float32)
            else:
                df[col] = df[col].astype(np.float64)    
    return df

df = downcast(df)
print(df.columns)


                   id  item_id  store_id  state_id  dept_id  cat_id     d  \
0  HOBBIES_1_001_CA_1     1438         1         1        4       2  1029   
1  HOBBIES_1_002_CA_1     1439         1         1        4       2  1029   
2  HOBBIES_1_003_CA_1     1440         1         1        4       2  1029   
3  HOBBIES_1_004_CA_1     1441         1         1        4       2  1029   
4  HOBBIES_1_005_CA_1     1442         1         1        4       2  1029   

   demand  lag_t7  rolling_mean_lag7_w7  ...  event_name_2  snap_CA  snap_TX  \
0     0.0     0.0              1.000000  ...             1        0        0   
1     0.0     1.0              0.142857  ...             1        0        0   
2     0.0     0.0              0.000000  ...             1        0        0   
3     3.0     0.0              1.142857  ...             1        0        0   
4     1.0     1.0              0.428571  ...             1        0        0   

   snap_WI sell_price  disaster_type  disaster_severity_

In [4]:
null_columns = df.columns[df.isna().any()].tolist()

# Kiểm tra giá trị 0
zero_columns = df.columns[(df == 0).any()].tolist()

print("Các cột có giá trị null:", null_columns)
print("Các cột có giá trị 0:", zero_columns)

Các cột có giá trị null: ['demand', 'lag_t7', 'rolling_mean_lag7_w7', 'rolling_mean_lag7_w28', 'sell_price', 'disaster_type']
Các cột có giá trị 0: ['demand', 'lag_t7', 'rolling_mean_lag7_w7', 'rolling_mean_lag7_w28', 'lag_t28', 'rolling_mean_lag28_w7', 'rolling_mean_lag28_w28', 'snap_CA', 'snap_TX', 'snap_WI', 'disaster_severity_flag', 'flag_ufc', 'gold_price']


In [ ]:
def fill_gold_price_with_rolling_mean(df):
    df = df.copy()
    
    for i in range(len(df)):
        # Kiểm tra nếu giá trị là null hoặc bằng 0
        if pd.isna(df.loc[i, 'gold_price']) or df.loc[i, 'gold_price'] == 0:
            if i == 0:  # Trường hợp đặc biệt: hàng đầu tiên
                # Đối với hàng đầu tiên, có thể sử dụng giá trị mặc định hoặc để nguyên
                continue  # Bỏ qua và giữ nguyên giá trị
            else:
                # Lấy tối đa 7 ngày trước để tính trung bình
                start_idx = max(0, i-7)
                # Lọc những giá trị hợp lệ (không null và khác 0)
                valid_values = [df.loc[j, 'gold_price'] for j in range(start_idx, i) 
                                if not pd.isna(df.loc[j, 'gold_price']) and df.loc[j, 'gold_price'] != 0]
                
                if valid_values:  # Nếu có ít nhất một giá trị hợp lệ
                    # Tính trung bình và gán
                    df.loc[i, 'gold_price'] = sum(valid_values) / len(valid_values)
                elif i > 0:  # Nếu không có giá trị hợp lệ nhưng có hàng trước đó
                    # Sử dụng giá trị của hàng liền trước (nếu có)
                    df.loc[i, 'gold_price'] = df.loc[i-1, 'gold_price']
    
    return df

# Gọi hàm
df = fill_gold_price_with_rolling_mean(df)

In [ ]:
df.to_parquet('../../data/train_val.parquet')

In [ ]:
df = df.drop(['date','disaster_type'],axis=1)
sample_submission = pd.read_csv('../../data/sample_submission.csv')
sample_submission = downcast(sample_submission)

LAGS = [7, 28]
WINDOWS = [7, 28]
FIRST = 1942 # first to predict
LENGTH = 28

def demand_features_eval(df):
    """ Same as demand_features but for the step-by-step evaluation """
    out = df.groupby('id', sort=False).last()
    for lag in LAGS:
        out[f'lag_t{lag}'] = df.groupby('id', sort=False)['demand'].nth(-lag-1).astype("float32")
        for w in WINDOWS:
            out[f'rolling_mean_lag{lag}_w{w}'] = df.groupby('id', sort=False)['demand'].nth(list(range(-lag-w, -lag))).groupby('id', sort=False).mean().astype("float32")
    
    return out.reset_index()

def prep_train(df,drop_d = 1000):
    
    x = list(set(df.columns) - {'id', 'd', 'demand'})
    print(x)
    # Split into test, valid, train
    test = df[df.d >= FIRST - max(LAGS) - max(WINDOWS) - 28]
    df = df[df.d < FIRST]

    xtrain, xvalid, ytrain, yvalid = train_test_split(df[x], df["demand"], test_size=0.1, shuffle=True, random_state=54)
    train = lgb.Dataset(xtrain, label = ytrain)
    valid = lgb.Dataset(xvalid, label = yvalid)
    return train, valid, test, x


def pred_all(fit, test, x):
    """ Calculate predictions """
    
    # Recursive prediction
    for i, day in enumerate(np.arange(FIRST, FIRST + LENGTH)):
        test_day = demand_features_eval(test[(test.d <= day) & (test.d >= day - max(LAGS) - max(WINDOWS))])
        test.loc[test.d == day, "demand"] = fit.predict(test_day[x])
    
    return test

def pred_to_csv(test, cols=sample_submission.columns, file="submission.csv"):
    """ Reshape predictions and save submission csv """
     
    # Prepare for reshaping
    test = test.assign(id=test.id + "_" + np.where(test.d < FIRST, "validation", "evaluation"),
                       F="F" + (test.d - FIRST + LENGTH + 1 - LENGTH * (test.d >= FIRST)).astype("str"))
    
    # Reshape
    submission = test.pivot(index="id", columns="F", values="demand").reset_index()[cols].fillna(1)
    
    # Export
    submission.to_csv(file, index=False)
    
    return True

train, valid, test, x = prep_train(df, 1000 - 28)

del df
gc.collect()

In [ ]:
import optuna 
def objective(trial, train_data, valid_data, num_boost_round=1500):
    """
    Định nghĩa hàm objective cho Optuna để tối ưu hóa hyperparameters của LightGBM
    """
    params = {
        'objective': 'poisson',
        'metric': 'rmse',
        'seed': 200,
        'force_row_wise': True,
        'boosting_type': 'gbdt',
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05),
        'lambda': trial.suggest_float('lambda', 0.01, 1.0, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 31, 127),
        'sub_row': trial.suggest_float('sub_row', 0.5, 0.9),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 10),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.9)
    }
    
    # Thêm các tham số khác nếu cần
    if trial.suggest_categorical('use_feature_fraction', [True, False]):
        params['feature_fraction'] = trial.suggest_float('feature_fraction', 0.5, 0.9)
    
    if trial.suggest_categorical('use_min_child_samples', [True, False]):
        params['min_child_samples'] = trial.suggest_int('min_child_samples', 5, 50)
    
    callbacks = [
        early_stopping(stopping_rounds=200),
        log_evaluation(period=100)
    ]
    
    # Tạo model
    model = lgb.train(
        params,
        train_data,
        num_boost_round=num_boost_round,
        valid_sets=[valid_data],
        callbacks=callbacks
    )
    
    # Lấy số vòng lặp tốt nhất
    best_iteration = model.best_iteration
    
    # Dự đoán trên validation set
    preds = model.predict(valid_data.get_data(), num_iteration=best_iteration)
    
    # Tính RMSE
    rmse = np.sqrt(mean_squared_error(valid_data.get_label(), preds))
    
    # Trả về RMSE và số vòng tốt nhất để lưu lại
    return rmse, best_iteration, model


def log_optuna_results(study, output_file="optuna_results.csv"):
    """
    Lưu tất cả kết quả của Optuna vào file CSV
    """
    results = []
    
    for trial in study.trials:
        if trial.state == optuna.trial.TrialState.COMPLETE:
            params = trial.params.copy()
            params['value'] = trial.value  # RMSE
            params['trial_number'] = trial.number
            
            # Thêm thời gian trial bắt đầu và kết thúc
            params['datetime_start'] = trial.datetime_start.strftime('%Y-%m-%d %H:%M:%S')
            params['datetime_complete'] = trial.datetime_complete.strftime('%Y-%m-%d %H:%M:%S')
            
            # Thêm thông tin khác từ user_attrs nếu có
            for key, value in trial.user_attrs.items():
                params[key] = value
            
            results.append(params)
    
    # Tạo DataFrame và lưu vào CSV
    results_df = pd.DataFrame(results)
    results_df.to_csv(output_file, index=False)
    print(f"Kết quả tối ưu hóa đã được lưu vào {output_file}")
    
    return results_df


def run_optuna_optimization(train, valid, test, x, sample_submission, n_trials=50):
    """
    Chạy quá trình tối ưu hóa với Optuna và lưu kết quả
    """
    # Tạo thư mục để lưu kết quả
    results_dir = "optimization_results"
    if not os.path.exists(results_dir):
        os.makedirs(results_dir)
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Khởi tạo study
    study = optuna.create_study(direction='minimize', 
                               study_name=f'lightgbm_optimization_{timestamp}')
    
    # Lưu thông tin về mỗi lần thử
    def objective_wrapper(trial):
        start_time = time.time()
        rmse, best_iteration, model = objective(trial, train, valid)
        
        # Lưu thêm thông tin về trial
        trial.set_user_attr('best_iteration', best_iteration)
        trial.set_user_attr('execution_time', time.time() - start_time)
        
        # Lưu model của trial hiện tại nếu đó là model tốt nhất
        if study.best_trial is None or rmse < study.best_value:
            model_path = os.path.join(results_dir, f"best_model_{timestamp}.txt")
            model.save_model(model_path)
            trial.set_user_attr('model_path', model_path)
        
        return rmse
    
    # Chạy tối ưu hóa
    print(f"Bắt đầu quá trình tối ưu hóa với {n_trials} trials...")
    study.optimize(objective_wrapper, n_trials=n_trials)
    
    # Lưu kết quả tối ưu hóa
    results_file = os.path.join(results_dir, f"optuna_results_{timestamp}.csv")
    results_df = log_optuna_results(study, results_file)
    
    # In ra thông tin về trial tốt nhất
    print(f"\nTrial tốt nhất #{study.best_trial.number}")
    print(f"Giá trị: {study.best_trial.value:.4f}")
    print("Tham số tốt nhất:")
    for key, value in study.best_trial.params.items():
        print(f"    {key}: {value}")
    
    # Lấy model tốt nhất
    best_model_path = study.best_trial.user_attrs.get('model_path')
    best_model = lgb.Booster(model_file=best_model_path)
    
    # Vẽ biểu đồ importance
    plt.figure(figsize=(10, 6))
    lgb.plot_importance(best_model, importance_type="gain", precision=0, height=0.5, figsize=(6, 10))
    plt.title('Feature Importance (Best Model)')
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, f"feature_importance_{timestamp}.png"))
    plt.show()
    
    # Vẽ biểu đồ lịch sử tối ưu hóa
    plt.figure(figsize=(10, 6))
    optuna.visualization.matplotlib.plot_optimization_history(study)
    plt.savefig(os.path.join(results_dir, f"optimization_history_{timestamp}.png"))
    plt.show()
    
    # Vẽ biểu đồ quan hệ tham số
    plt.figure(figsize=(12, 10))
    optuna.visualization.matplotlib.plot_param_importances(study)
    plt.savefig(os.path.join(results_dir, f"param_importances_{timestamp}.png"))
    plt.show()
    
    # Tạo biểu đồ song song
    try:
        optuna.visualization.matplotlib.plot_parallel_coordinate(study)
        plt.savefig(os.path.join(results_dir, f"parallel_coordinate_{timestamp}.png"))
        plt.show()
    except:
        print("Không thể tạo biểu đồ song song coordinate.")
    
    # Dự đoán với model tốt nhất và lưu submission
    print("\nDự đoán với model tốt nhất...")
    preds = pred_all(best_model, test, x)
    submission_file = os.path.join(results_dir, f"submission_{timestamp}.csv")
    pred_to_csv(preds, cols=sample_submission.columns, file=submission_file)
    
    # Lưu thông tin về tham số tốt nhất
    best_params = study.best_trial.params
    best_params_df = pd.DataFrame([best_params])
    best_params_df.to_csv(os.path.join(results_dir, f"best_params_{timestamp}.csv"), index=False)
    
    return study, best_model, results_df

In [ ]:
study, best_model, results_df = run_optuna_optimization(train, valid, test, x, sample_submission, n_trials=1)

# Bạn có thể sử dụng best_model trực tiếp thay vì fit trong hàm fit_model
print("\nDự đoán với model tốt nhất và lưu submission cuối cùng...")
preds = pred_all(best_model, test, x)
pred_to_csv(preds, cols=sample_submission.columns, file="submission_final.csv")

In [ ]:
def fit_model(train, valid, num_r):
    """ Fit LightGBM model """
    
    params = {
        'metric': 'rmse',
        'objective': 'poisson',
        'seed': 200,
        'force_row_wise': True,
        'boosting_type': 'gbdt',
        'learning_rate': 0.015,
        'lambda': 0.1,
        'num_leaves': 63,
        'sub_row': 0.7,
        'bagging_freq': 1,
        'colsample_bytree': 0.7
    }

    callbacks = [
        early_stopping(stopping_rounds=200),
        log_evaluation(period=100)
    ]

    fit = lgb.train(params,
                    train,
                    num_boost_round=num_r,
                    valid_sets=[valid],
                    callbacks=callbacks)

    lgb.plot_importance(fit, importance_type="gain", precision=0, height=0.5, figsize=(6, 10))
    plt.show()

    return fit


fit = fit_model(train, valid,1500)
pred = pred_all(fit, test, x)
pred_to_csv(pred, cols=sample_submission.columns, file="submission.csv")